# SMS Spam Detection — Model Training and Evaluation

This notebook trains and evaluates an SMS spam classifier. Download the SMS Spam Collection dataset and place it in this folder as `spam.csv` before running.

Expected dataset fields: `v1` (label: ham/spam) and `v2` (message text).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import pickle


In [ ]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df.head()

In [ ]:
# Adapt to common SMS Spam Collection dataset columns
df = df[['v1', 'v2']].rename(columns={'v1':'label', 'v2':'message'})
df = df.dropna()
df['label'] = df['label'].str.strip().str.lower()
df = df[df['label'].isin(['ham','spam'])]
df['label'].value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['message'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1,2))),
    ('classifier', MultinomialNB())
])
pipeline.fit(X_train, y_train)


In [ ]:
predictions = pipeline.predict(X_test)
print('Accuracy:', accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions, labels=['ham','spam']))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, labels=['ham','spam'])
plt.title('SMS Spam Classifier — Confusion Matrix')
plt.show()

In [ ]:
# Save the full pipeline so app.py can accept raw message text
with open('model.pkl', 'wb') as file:
    pickle.dump(pipeline, file)
print('Saved full preprocessing + classifier pipeline to model.pkl')

In [ ]:
pipeline.predict(['Congratulations! You won a prize. Click the link now.'])

## Notes
- Run all cells to create a fresh `model.pkl` pipeline.
- Only load pickle files from trusted sources.
- Report test metrics from your own run; do not invent or copy metrics.